In [2]:
from pathlib import Path
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42

DATA_PATHS = [
    Path("../data/raw/laptops_cleaned.csv"),
    Path("data/raw/laptops_cleaned.csv"),
]
# Try to find the data file in the specified paths
data_path = next((p for p in DATA_PATHS if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not find laptops_cleaned.csv in the expected locations.")

df = pd.read_csv(data_path)

target = "price"
# Define the feature set (ex the target variable)
feature_set_a = [
    "brand",
    "device_category",
    "rating",
    "cpu_brand",
    "cpu_family",
    "cpu_core_count",
    "cpu_thread_count",
    "ram_gb",
    "storage_gb",
    "display_width_px",
    "display_height_px",
    "display_size_inch",
    "gpu_brand",
    "gpu_type",
    "gpu_vram_gb",
    "os_name",
    "warranty_years",
]
# Check for missing columns in the DataFrame
missing_columns = sorted(set(feature_set_a + [target]) - set(df.columns))
if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")
# Prepare feature matrix X and target vector y
X = df[feature_set_a].copy()
y = df[target].copy()
# Print the shapes of X and y, and the feature columns
print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nFeature columns:")
print(X.columns.tolist())

X shape: (992, 17)
y shape: (992,)

Feature columns:
['brand', 'device_category', 'rating', 'cpu_brand', 'cpu_family', 'cpu_core_count', 'cpu_thread_count', 'ram_gb', 'storage_gb', 'display_width_px', 'display_height_px', 'display_size_inch', 'gpu_brand', 'gpu_type', 'gpu_vram_gb', 'os_name', 'warranty_years']


In [ ]:
# Identify numeric and categorical features
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

In [ ]:
# Check for missing values in the feature matrix X
X.isnull().sum().sort_values(ascending=False)

In [ ]:
# Define preprocessing pipelines for numeric and categorical features
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

numeric_transformer_linear = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

numeric_transformer_tree = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])
# Combine preprocessing steps into ColumnTransformers for linear and tree-based models
preprocessor_linear = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_linear, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)
# The tree based preprocessor doesnt include scaling for numeric features
preprocessor_tree = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_tree, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)
# Print preprocessors to verify structure
print("Preprocessors created successfully.")

In [ ]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

In [ ]:
# Apply the linear preprocessor to the training and testing data
X_train_linear = preprocessor_linear.fit_transform(X_train)
X_test_linear = preprocessor_linear.transform(X_test)

# Check on the shapes after transforming
print("Transformed train shape:", X_train_linear.shape)
print("Transformed test shape :", X_test_linear.shape)

In [ ]:
# Get feature names after transformation for linear preprocessor
feature_names_linear = preprocessor_linear.get_feature_names_out()
pd.Series(feature_names_linear, name="transformed_feature").head(30)